# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Davionnic/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Task type: ranking / scoring** (supervised priority score over pages).

Lane stays **Refresh / Content Opportunity Scoring** (same as ML-02). The decision is "which page should a content / SEO reviewer open next?" — not "will Google rank this higher if we edit it?"

Classification alone ("declining: yes/no") is a building block, but the product output reviewers need is a **ranked queue**: many pages, tiny capacity, so order matters. Clustering would invent groups without a review priority. Pure signal analysis is useful later for audits; it does not ship a queue.

So the ML task is: score each page for review priority, sort descending, evaluate with **Precision@K**. The binary proxy label below trains/scores that ranker; the deployed artifact is the ranked list + reason codes, not a hard auto-edit.

In [1]:
import os
import numpy as np
import pandas as pd

# walk up until we find the repo root (works from work/notebooks/ or Colab)
while not os.path.isdir("data/raw") and os.getcwd() != "/":
    os.chdir("..")

RAW = "data/raw/content_refresh_anonymized.csv"
assert os.path.exists(RAW), f"missing starter CSV at {os.path.abspath(RAW)}"
df = pd.read_csv(RAW)

print("Working dir:", os.getcwd())
print(f"starter rows={len(df):,}  cols={df.shape[1]}  clients={df['client_id'].nunique()}")
print("task type locked: ranking/scoring -> Precision@K review queue")
print("lane (from ML-02): Refresh / Content Opportunity Scoring")
print("NOT this week: clustering, causal SEO automation, HF warehouse")

Working dir: /workspace/flyrank-ml-internship
starter rows=30,000  cols=44  clients=32
task type locked: ranking/scoring -> Precision@K review queue
lane (from ML-02): Refresh / Content Opportunity Scoring
NOT this week: clustering, causal SEO automation, HF warehouse


## 2. Target or proxy

**Proxy label (starter):** `is_declining_label = 1` when `trend_direction == "down"`, else `0`.

Where it comes from: a **defined bucket** on the same trailing-90-day window as the features — not a later observed outcome. The pipeline writes this in `01_prepare_features.py`. Because the label is derived from `trend_direction` (itself from `trend_pct`), both `trend_direction` and `trend_pct` are **never features**.

Honest framing:
- This is a **teaching proxy**, good enough to train a ranker and compare it to a hand rule on the 30k starter.
- It is **not** proof that the page will keep declining, or that a refresh will recover traffic.
- A stronger later label (warehouse, if/when available): features from a prior window → decline/recovery in a **future** window under client-holdout.

What the model outputs for humans: a priority score (probability of the proxy + blended baseline), not a causal "do this edit" command.

In [2]:
# Target / proxy from the starter slice — observed counts only
y = (df["trend_direction"].astype(str).str.lower() == "down").astype(int)
df = df.copy()
df["is_declining_label"] = y

n = len(df)
pos = int(y.sum())
print("--- proxy target ---")
print("definition: is_declining_label = (trend_direction == 'down')")
print(f"positives: {pos:,} / {n:,} = {pos/n:.1%}  (base rate)")
print("label-source columns (FORBIDDEN as features): trend_direction, trend_pct")
print("IDs (split/join only, never features): content_id, client_id")
print("claim mode: proxy label for decision-support ranking — not a future outcome")

--- proxy target ---
definition: is_declining_label = (trend_direction == 'down')
positives: 16,262 / 30,000 = 54.2%  (base rate)
label-source columns (FORBIDDEN as features): trend_direction, trend_pct
IDs (split/join only, never features): content_id, client_id
claim mode: proxy label for decision-support ranking — not a future outcome


## 3. Success metric

**Primary metric: Precision@50** on a **client-holdout** split.

Plain English: of the 50 pages the system puts at the top of the review queue, what fraction carry the proxy decline label? That matches the decision — a reviewer can only open ~K pages today.

Why Precision@K, not accuracy:
- Base rate on this starter is **~54%** declining. Accuracy of "always say down" looks fine and is useless for ranking.
- ROC-AUC / average precision are secondary diagnostics for ranking quality; selection for the starter pipeline is Precision@50.

What "good" means here (directional, measured — not invented):
- Beat a transparent hand-rule baseline on the **same** holdout and K.
- Report the number from a real run; do not round a story around a target you picked after seeing scores.

Wrong-call costs still apply: high Precision@K wastes fewer editor hours on healthy pages; missing true declines (false negatives deeper in the list) is the other side of the tradeoff.

In [3]:
def precision_at_k(y_true, scores, k: int) -> float:
    y_true = np.asarray(y_true)
    scores = np.asarray(scores, dtype=float)
    k = min(int(k), len(y_true))
    if k <= 0:
        return float("nan")
    top = np.argsort(-scores)[:k]
    return float(y_true[top].mean())

base_rate = float(y.mean())
# single-signal rankings on FULL sample (not holdout) — sanity checks only
p50_imp = precision_at_k(y, df["impressions_90d"].fillna(0), 50)
p50_age = precision_at_k(y, df["content_age_days"].fillna(0), 50)
p50_fresh = precision_at_k(y, df["days_since_last_update"].fillna(0), 50)
p50_neg_ctr = precision_at_k(y, -df["ctr"].fillna(0), 50)

print("--- success metric framing (starter CSV, full-sample sanity — NOT holdout) ---")
print(f"base rate (random / always-down floor for ranking): {base_rate:.3f}")
print(f"Precision@50 rank-by-impressions_90d:        {p50_imp:.3f}")
print(f"Precision@50 rank-by-content_age_days:       {p50_age:.3f}")
print(f"Precision@50 rank-by-days_since_last_update: {p50_fresh:.3f}")
print(f"Precision@50 rank-by-lowest-ctr:             {p50_neg_ctr:.3f}")
print("primary metric for later models: Precision@50 under client-holdout")
print("secondary: ROC-AUC, average precision — diagnostics, not the decision metric")
print("note: full-sample single-signal P@K can look strong (age) and still fail holdout / product use")

--- success metric framing (starter CSV, full-sample sanity — NOT holdout) ---
base rate (random / always-down floor for ranking): 0.542
Precision@50 rank-by-impressions_90d:        0.420
Precision@50 rank-by-content_age_days:       0.840
Precision@50 rank-by-days_since_last_update: 0.560
Precision@50 rank-by-lowest-ctr:             0.480
primary metric for later models: Precision@50 under client-holdout
secondary: ROC-AUC, average precision — diagnostics, not the decision metric
note: full-sample single-signal P@K can look strong (age) and still fail holdout / product use


## 4. The unit of analysis, as a real dataframe

**One row = one pseudonymized content page** (`content_id`) for one client (`client_id`), with trailing-90-day search/engagement signals.

Grain check on the starter: 30,000 rows, 30,000 distinct `content_id`, 32 clients. No URL/title/query text in this file.

Feature families I will allow later (honest list, no label leak):
- **Numeric:** volume/competition/CPC, length, log traffic, days-with-activity, age/freshness, CTR, position, engagement/scroll, AI traffic share (same family as `MODEL_NUMERIC_FEATURES` in `scripts/ml_utils.py`).
- **Categorical tiers:** competition level, content type, intent, age/freshness/word/impression/position tiers.
- **Never features:** `trend_direction`, `trend_pct`, raw IDs as model inputs, anything that looks like a client name/URL/query.

In [4]:
# Unit of analysis: show the grain + a safe preview (pseudonyms only)
assert df["content_id"].nunique() == len(df), "expected one row per content_id"
preview_cols = [
    "content_id", "client_id", "content_type", "impressions_90d", "avg_position",
    "ctr", "content_age_days", "days_since_last_update", "engagement_rate",
    "word_count", "age_tier", "impression_tier", "is_declining_label",
]
preview_cols = [c for c in preview_cols if c in df.columns]

print("--- unit of analysis ---")
print(f"one row = one content page | rows={len(df):,} | unique content_id={df['content_id'].nunique():,}")
print(f"clients (pseudonym): {df['client_id'].nunique()}")
print("preview (no URLs/titles/queries in this CSV):")
display_df = df[preview_cols].head(5)
print(display_df.to_string(index=False))

# feature candidates present now (raw) vs engineered later by pipeline
raw_numeric = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d",
    "days_with_impressions", "days_with_sessions", "content_age_days",
    "days_since_last_update", "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct",
]
raw_cats = [
    "competition_level", "content_type", "main_intent",
    "age_tier", "freshness_tier", "word_count_tier", "impression_tier", "position_tier",
]
print("raw numeric present:", sum(c in df.columns for c in raw_numeric), "/", len(raw_numeric))
print("raw categorical present:", sum(c in df.columns for c in raw_cats), "/", len(raw_cats))
print("forbidden-as-features check:", [c for c in ("trend_direction", "trend_pct") if c in df.columns])

--- unit of analysis ---
one row = one content page | rows=30,000 | unique content_id=30,000
clients (pseudonym): 32
preview (no URLs/titles/queries in this CSV):
          content_id         client_id    content_type  impressions_90d  avg_position  ctr  content_age_days  days_since_last_update  engagement_rate  word_count age_tier impression_tier  is_declining_label
content_304f48230142 client_f369cb89fc keyword article             3803          10.6 0.76               187                      20             5.88      3221.0  181-365            good                   1
content_a1fb4e703a9e client_4e07408562 keyword article            15320          20.3 0.05               445                      25             0.00      2481.0     365+            good                   1
content_9aa793d4d895 client_7f2253d7e2 keyword article            12581          36.5 0.09               141                      20             0.00      3515.0   91-180            good                   1
content_3

## 5. Why ML beats a fixed rule here

A fixed if-statement *can* surface obvious cases (e.g. stale + high impressions). It fails when many weak, interacting signals decide the middle of the queue — where most review capacity actually goes.

Evidence from this starter slice (full-sample counts, not a holdout model claim):
- Hand-style flags fire at very different rates and decline rates (some rare+sharp, some mass+noisy).
- Decline rate is **not** a clean step function of "how many flags fired" — more flags is not always more decline.
- Decline rate by `age_tier × impression_tier` is a messy grid, not one threshold.
- Single-signal rankings disagree with each other (impressions vs age vs freshness vs CTR).

That is the bar for ML: a learned score that combines many observables under client-holdout and **beats** the transparent baseline on Precision@K. If it cannot beat the rule, keep the rule.

In [5]:
# Messiness check: fixed flags vs interacting tiers (full sample)
flags = pd.DataFrame({
    "stale_visible": (df["days_since_last_update"].fillna(0) >= 180) & (df["impressions_90d"] >= 500),
    "thin_visible": (df["word_count"].fillna(0) > 0) & (df["word_count"] < 1200) & (df["impressions_90d"] >= 250),
    "page_one_decay_risk": (df["avg_position"] > 0) & (df["avg_position"] <= 10) & (df["content_age_days"] >= 180),
    "low_ctr_visible": (
        (df["impressions_90d"] >= 500) & (df["avg_position"] > 0) & (df["avg_position"] <= 20) & (df["ctr"] < 0.5)
    ),
    "low_eng_visible": (
        (df["sessions_90d"] >= 30)
        & ((df["engagement_rate"].fillna(100) < 30) | (df["scroll_rate"].fillna(100) < 30))
    ),
})

print("--- hand-style flags (no trend_* in the flag definitions below) ---")
for col in flags.columns:
    m = flags[col]
    n_flag = int(m.sum())
    rate = float(y[m].mean()) if n_flag else float("nan")
    print(f"{col:22s} n={n_flag:5d}  decline_rate_among_flagged={rate:.3f}")

nflags = flags.sum(axis=1)
print("\ndecline rate by #flags fired (should be monotonic if a simple count-rule worked — it is not):")
for k in sorted(nflags.unique()):
    mask = nflags == k
    print(f"  flags={k}: n={int(mask.sum()):5d}  decline_rate={float(y[mask].mean()):.3f}")

# additive score WITHOUT trend_direction (honest features only)
hand_score = (
    2 * flags["stale_visible"].astype(int)
    + 2 * flags["thin_visible"].astype(int)
    + 2 * flags["page_one_decay_risk"].astype(int)
    + 1 * flags["low_ctr_visible"].astype(int)
    + 1 * flags["low_eng_visible"].astype(int)
    + np.log1p(df["impressions_90d"].fillna(0)) / 10.0
    + df["content_age_days"].fillna(0) / 1000.0
)
print("\nadditive hand score Precision@K (FULL sample sanity, not client-holdout):")
for k in (20, 50, 100):
    print(f"  Precision@{k}: {precision_at_k(y, hand_score, k):.3f}")

print("\ndecline rate grid: age_tier x impression_tier (messy surface):")
grid = pd.crosstab(df["age_tier"], df["impression_tier"], values=y, aggfunc="mean")
print(grid.round(3).to_string())
print("\nwhy ML: many overlapping weak signals + non-monotonic flag counts + tier interactions")
print("bar to clear later: beat transparent baseline on Precision@50 with client-holdout")

--- hand-style flags (no trend_* in the flag definitions below) ---
stale_visible          n=   17  decline_rate_among_flagged=0.941
thin_visible           n=   82  decline_rate_among_flagged=0.537
page_one_decay_risk    n= 7076  decline_rate_among_flagged=0.518
low_ctr_visible        n= 9759  decline_rate_among_flagged=0.627
low_eng_visible        n= 7113  decline_rate_among_flagged=0.546

decline rate by #flags fired (should be monotonic if a simple count-rule worked — it is not):
  flags=0: n=13392  decline_rate=0.498
  flags=1: n=10532  decline_rate=0.581
  flags=2: n= 4713  decline_rate=0.594
  flags=3: n= 1363  decline_rate=0.492

additive hand score Precision@K (FULL sample sanity, not client-holdout):
  Precision@20: 0.800
  Precision@50: 0.580
  Precision@100: 0.480

decline rate grid: age_tier x impression_tier (messy surface):
impression_tier  excellent   good    low  moderate
age_tier                                          
181-365              0.483  0.576  0.396     0.6

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

**Frame (one paragraph):** For a content/SEO reviewer deciding which page to open next, we will build a ranked review queue from the anonymized starter page table, scoring the proxy `is_declining_label` (`trend_direction == "down"`) and measuring **Precision@50** under client-holdout. A wrong call wastes editor hours or buries a real decline. A plain rule is not enough because flags and tiers interact non-monotonically. Claims stay observed / directional / decision-support — not causal SEO recovery.